<center><h1> NON PR Streak </h1></center>

The following code is a slightly modified version of one written by [Thomas Stokes](https://github.com/ThomasStokes1998), WCAID [2017STOK03](https://www.worldcubeassociation.org/persons/2017STOK03). Full credit goes to him.

It calculates the longest streak of consecutive WCA competitions without a <b>personal record</b> (PR) for each competitor.

<small>Startcomp is the first competition of the streak.<br>Endcomp is the competition that breaks the streak (N+1).</small>

### Imports & functions

In [1]:
import pandas as pd
from datetime import date
import numpy as np

#file download libraries
import urllib.request
import zipfile

from datetime import datetime

url = "https://www.worldcubeassociation.org/export/results/WCA_export.tsv.zip"
extract_dir = "Tables"

zip_path, _ = urllib.request.urlretrieve(url)
with zipfile.ZipFile(zip_path, "r") as f:
    f.extractall(extract_dir)

results_path = r'Tables/WCA_export_Results.tsv'
competitions_path = r'Tables/WCA_export_Competitions.tsv'

results = pd.read_csv(results_path, delimiter='\t')
competitions = pd.read_csv(competitions_path, delimiter='\t')


In [2]:
def mbfPoints(p: int, oldstyle: bool=False) -> float:
    if oldstyle:
        tt = p % 100_000
        ss = 199 - p // 10_000_000
        aa = (p // 100_000) % 100
        if tt == 99_999:
            tt = min(600 * aa, 3_600)
        return ss - aa + 1 - tt / 3_600
    else:
        dd = 99 - p // 10_000_000
        mm = p % 100
        tt = (p // 100) % 100_000
        aa = dd + 2 * mm
        if tt == 99_999:
            tt = min(600 * aa, 3_600)
        if aa < 6:
            return dd + 1 - tt / (600 * aa)
        else:
            return dd + 1 - tt / 3_600


### Dictionaries to keep track of all the data

In [3]:
compdates = {"ongoing":date.today()}
for i, comp in enumerate(competitions.id):
    compdates[comp] = date(competitions.year[i], competitions.month[i], competitions.day[i])


In [4]:
personnames = {}
maxstreaks = {}
maxstart = {}
maxend = {}
currentcomp = {}
startcomps = {}
currentstreak = {}
gotpr = {}
personprs = {}

### Code

In [5]:
def updatedicts(event: str, wcaid: str, single: int, average: int):
    # Add a new event 
    if event not in personprs[wcaid]:
        if event == "333mbf":
            personprs[wcaid][event] = 0
        elif event == "333mbo":
            personprs[wcaid][event] = 0
        else:
            personprs[wcaid][event] = [360_000, 360_000]
    # Check if a new PR has been set
    if event == "333mbf":
        if single > 0 and personprs[wcaid][event] <= mbfPoints(single):
            personprs[wcaid][event] = mbfPoints(single)
            gotpr[wcaid] = True
    elif event == "333mbo":
        if single > 0 and personprs[wcaid][event] <= mbfPoints(single, True):
            personprs[wcaid][event] = mbfPoints(single, True)
            gotpr[wcaid] = True
    else:
        if single > 0 and personprs[wcaid][event][0] >= single:
            personprs[wcaid][event][0] = single
            gotpr[wcaid] = True
        if average > 0 and personprs[wcaid][event][1] >= average:
            personprs[wcaid][event][1] = average
            gotpr[wcaid] = True

Change the country in the first row below to the country of choice or to "world" for global PR straks

In [6]:
def main_old(country: str = "World"):
    if country == "World":
        res = results
    else:
        res = results[results.personCountryId == country].reset_index(drop="index")
    # Order the res dataframe by date
    res["date"] = res["competitionId"].apply(lambda x: compdates[x])
    res = res.sort_values("date").reset_index(drop="index")
    for i, wcaid in enumerate(res.personId):
        comp = res.competitionId[i]
        event = res.eventId[i]
        single = res.best[i]
        average = res.average[i]
        if wcaid not in personnames:
            # Initial values for a new id
            personnames[wcaid] = res.personName[i]
            maxstreaks[wcaid] = 0
            maxstart[wcaid] = "none"
            maxend[wcaid] = "none"
            startcomps[wcaid] = "none"
            currentcomp[wcaid] = comp
            currentstreak[wcaid] = 0
            gotpr[wcaid] = False
            personprs[wcaid] = {}
        if currentcomp[wcaid] == comp:
            updatedicts(event, wcaid, single, average)
        else:
            if gotpr[wcaid]:
                c = currentstreak[wcaid]
                if c > maxstreaks[wcaid]:
                    maxstreaks[wcaid] = c
                    maxstart[wcaid] = startcomps[wcaid]
                    maxend[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] = 0
            else:
                if currentstreak[wcaid] == 0:
                    startcomps[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] += 1
            # Reset values for a new competition
            gotpr[wcaid] = False
            currentcomp[wcaid] = comp
            updatedicts(event, wcaid, single, average)
    # Update ongoing streaks
    for wcaid in gotpr:
        if maxstart[wcaid] == "none":
            maxstart[wcaid] = currentcomp[wcaid]
            maxend[wcaid] = currentcomp[wcaid]
        if gotpr[wcaid]:
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                maxstart[wcaid] = startcomps[wcaid]
                maxend[wcaid] = currentcomp[wcaid]
        else:
            if currentstreak[wcaid] == 0:
                startcomps[wcaid] = currentcomp[wcaid]
            currentstreak[wcaid] += 1
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                maxstart[wcaid] = startcomps[wcaid]
                maxend[wcaid] = "ongoing"
            
                
    # Create the dataframe
    names = [personnames[wcaid] for wcaid in gotpr]
    maxstreak = [maxstreaks[wcaid] for wcaid in gotpr]
    maxstarts = [maxstart[wcaid] for wcaid in gotpr]
    maxends = [maxend[wcaid] for wcaid in gotpr]
    maxdays = [(compdates[maxends[i]] - compdates[maxstarts[i]]).days for i, _ in enumerate(maxends)]
    df = pd.DataFrame({"name":names, "NonPrStreak":maxstreak, "startcomp":maxstarts, "endcomp":maxends, 
    "days":maxdays})
    df = df.sort_values("NonPrStreak", ascending=False).reset_index(drop="index")
    df.to_csv(f"maxnonprstreak{country}.csv", index=False)


In [7]:
def main(country: str = "Italy"):
    if country == "World":
        res = results
    else:
        res = results[results.personCountryId == country].reset_index(drop=True)

    # Order the res dataframe by date
    res["date"] = res["competitionId"].apply(lambda x: compdates[x])
    res = res.sort_values("date").reset_index(drop=True)

    for i, wcaid in enumerate(res.personId):
        comp = res.competitionId[i]
        event = res.eventId[i]
        single = res.best[i]
        average = res.average[i]

        if wcaid not in personnames:
            # Initial values for a new id
            personnames[wcaid] = res.personName[i]
            maxstreaks[wcaid] = 0
            maxstart[wcaid] = "none"
            maxend[wcaid] = "none"
            startcomps[wcaid] = "none"
            currentcomp[wcaid] = comp
            currentstreak[wcaid] = 0
            gotpr[wcaid] = False
            personprs[wcaid] = {}

        if currentcomp[wcaid] == comp:
            updatedicts(event, wcaid, single, average)
        else:
            if gotpr[wcaid]:
                c = currentstreak[wcaid]
                if c > maxstreaks[wcaid]:
                    maxstreaks[wcaid] = c
                    # Ensure correct order
                    if compdates[startcomps[wcaid]] <= compdates[currentcomp[wcaid]]:
                        maxstart[wcaid] = startcomps[wcaid]
                        maxend[wcaid] = currentcomp[wcaid]
                    else:
                        maxstart[wcaid] = currentcomp[wcaid]
                        maxend[wcaid] = startcomps[wcaid]
                currentstreak[wcaid] = 0
            else:
                if currentstreak[wcaid] == 0:
                    startcomps[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] += 1

            # Reset for the new comp
            gotpr[wcaid] = False
            currentcomp[wcaid] = comp
            updatedicts(event, wcaid, single, average)

    # Update ongoing streaks
    for wcaid in gotpr:
        if maxstart[wcaid] == "none":
            maxstart[wcaid] = currentcomp[wcaid]
            maxend[wcaid] = currentcomp[wcaid]

        if gotpr[wcaid]:
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                if compdates[startcomps[wcaid]] <= compdates[currentcomp[wcaid]]:
                    maxstart[wcaid] = startcomps[wcaid]
                    maxend[wcaid] = currentcomp[wcaid]
                else:
                    maxstart[wcaid] = currentcomp[wcaid]
                    maxend[wcaid] = startcomps[wcaid]
        else:
            if currentstreak[wcaid] == 0:
                startcomps[wcaid] = currentcomp[wcaid]
            currentstreak[wcaid] += 1
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                if compdates[startcomps[wcaid]] <= compdates[currentcomp[wcaid]]:
                    maxstart[wcaid] = startcomps[wcaid]
                    maxend[wcaid] = "ongoing"
                else:
                    maxstart[wcaid] = currentcomp[wcaid]
                    maxend[wcaid] = "ongoing"

    # Create the dataframe
    names = [personnames[wcaid] for wcaid in gotpr]
    maxstreak = [maxstreaks[wcaid] for wcaid in gotpr]
    maxstarts = [maxstart[wcaid] for wcaid in gotpr]
    maxends = [maxend[wcaid] for wcaid in gotpr]

    maxdays = []
    today = datetime.today().date()

    for start, end in zip(maxstarts, maxends):
        if start not in compdates:
            maxdays.append(None)
        elif end == "ongoing":
            delta = (today - compdates[start]).days
            maxdays.append(max(delta, 0))
        elif end in compdates:
            delta = (compdates[end] - compdates[start]).days
            maxdays.append(max(delta, 0))
        else:
            maxdays.append(None)

    df = pd.DataFrame({
        "name": names,
        "NonPrStreak": maxstreak,
        "startcomp": maxstarts,
        "endcomp": maxends,
        "days": maxdays
    })

    df = df.sort_values("NonPrStreak", ascending=False).reset_index(drop=True)
    df.to_csv(f"maxnonprstreak{country}.csv", index=False)


### Result

In [8]:
if __name__ == "__main__":
    # Longest PR Streak in Country
    main()

Pr streaks

In [9]:
a = pd.read_csv('maxnonprstreakWorld.csv') #change the name if you change the country to 'maxprstreak[country].csv'
a.index += 1
a.head(20)

,name,NonPrStreak,startcomp,endcomp,days
1,Charles Zhu (朱彦臣),45,YangzhouOpen2017,NanaimoBacktoSchool2022,1854
2,Matthew Dickman,41,OrangeCountyNewcomers2024,ongoing,484
3,Natán Riggenbach,39,SpeedsolvingPucallpa2016,JaenMountainCubes2018,980
4,Joshua Feran,34,7x7MadisonPark2022,ongoing,1009
5,Yulia Sidorova,33,SuisseToyFastFingers2017,ongoing,2816
6,Dave Campbell,33,HillsdaleWinter2012,LondonLimitedSpring2017,1848
7,Jacob Ambrose,31,StauntonSpeedsolving2023,ongoing,653
8,Nicolò Simone,30,MilanCubeDays2015,ongoing,3572
9,Karolina Wiącek,28,PolishNationals2014,GdanskRubiksCubeDay2016,889
10,Shelley Chang,28,USNationals2012,CubingSkillcon2015,1191


Ongoing PR streaks

In [10]:
a[a['endcomp']=='ongoing'].reset_index(drop=True).head(20)

,name,NonPrStreak,startcomp,endcomp,days
0,Matthew Dickman,41,OrangeCountyNewcomers2024,ongoing,484
1,Joshua Feran,34,7x7MadisonPark2022,ongoing,1009
2,Yulia Sidorova,33,SuisseToyFastFingers2017,ongoing,2816
3,Jacob Ambrose,31,StauntonSpeedsolving2023,ongoing,653
4,Nicolò Simone,30,MilanCubeDays2015,ongoing,3572
5,Michał Bogdan,24,BrizZonSideOpenII2023,ongoing,793
6,Donglei Li (李冬雷),24,XianCherryBlossom2017,ongoing,3011
7,Ryan DeLine,23,PleaseBeAsQuietAsACat2018,ongoing,2655
8,Laura Ohrndorf,22,FrankfurtCubeDays2020,ongoing,1955
9,Ming Zheng (郑鸣),22,TianjinSummerSolstice2021,ongoing,1471
